# 🏨 Hotel Booking Cancellation — Machine Learning Pipeline
### Snipers Team | Data Mining & Visualization Course Project
---
**Target variable:** `is_canceled`  
**Models:** Random Forest · Logistic Regression · XGBoost · Decision Tree  
**Goal:** Predict booking cancellations at booking time and export probabilities for the live dashboard.

---


## 1. Imports & Configuration

In [ ]:
# ─── Core ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ─── Sklearn ─────────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline

# ─── Optional XGBoost (install if needed) ────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️  XGBoost not installed — skipping (run: pip install xgboost)")

# ─── Plot style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.family': 'DejaVu Sans',
})

PALETTE   = ['#1B6CA8', '#E63946', '#2EC4B6', '#F4A261', '#2D6A4F']
CANCEL_C  = {'Not Canceled': '#1B6CA8', 'Canceled': '#E63946'}

print("✅ All imports successful")


## 2. Load & Preview Dataset
> **Note:** Upload `hotel_booking.csv` to the same directory, or mount Google Drive and update the path below.


In [ ]:
# ─── Load ────────────────────────────────────────────────────────────────
# If running on Google Colab, uncomment the next two lines:
# from google.colab import drive
# drive.mount('/content/drive')

DATA_PATH = 'hotel_booking.csv'   # ← update path if needed

df_raw = pd.read_csv(DATA_PATH)

print(f"Dataset shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Cancellation % : {df_raw['is_canceled'].mean()*100:.1f}%")
print(f"Hotels         : {df_raw['hotel'].unique().tolist()}")
print(f"Years          : {sorted(df_raw['arrival_date_year'].unique().tolist())}")
df_raw.head(3)


## 3. Data Cleaning & Feature Engineering

In [ ]:
# ─── Copy raw ───────────────────────────────────────────────────────────
df = df_raw.copy()

# ─── Drop high-sparsity column ───────────────────────────────────────────
df.drop(columns=['company'], inplace=True, errors='ignore')

# ─── Fill missing values ─────────────────────────────────────────────────
df['agent']    = df['agent'].fillna(0).astype(int)
df['country']  = df['country'].fillna('Unknown')
df['children'] = df['children'].fillna(0).astype(int)

# ─── Remove invalid records ───────────────────────────────────────────────
df = df[(df['adults'] + df['children'] + df['babies']) > 0]  # ghost bookings
df = df[df['adr'] >= 0]                                        # negative rate
df = df[df['adults'] <= 4]                                     # extreme outlier

# ─── Cap ADR at 99th percentile (Winsorize) ──────────────────────────────
adr_99 = df['adr'].quantile(0.99)
df['adr'] = df['adr'].clip(upper=adr_99)

# ─── Feature Engineering ─────────────────────────────────────────────────
df['total_stay']    = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['total_guests']  = df['adults'] + df['children'] + df['babies']
df['has_children']  = (df['children'] + df['babies'] > 0).astype(int)
df['booking_month'] = pd.to_datetime(
    df['arrival_date_year'].astype(str) + '-' +
    df['arrival_date_month'].astype(str) + '-01'
).dt.month
df['is_weekend_heavy'] = (df['stays_in_weekend_nights'] > df['stays_in_week_nights']).astype(int)
df['repeat_cancel_risk'] = (df['previous_cancellations'] > 0).astype(int)
df['revenue_potential']  = df['adr'] * df['total_stay']

print(f"Cleaned shape  : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"New features   : total_stay, total_guests, has_children, booking_month,")
print(f"                 is_weekend_heavy, repeat_cancel_risk, revenue_potential")
df[['total_stay','total_guests','revenue_potential','repeat_cancel_risk']].describe().round(2)


## 4. Feature Selection & Encoding

In [ ]:
# ─── Select features ────────────────────────────────────────────────────
FEATURES = [
    # Numeric
    'lead_time', 'adr', 'total_stay', 'total_guests',
    'booking_changes', 'days_in_waiting_list', 'previous_cancellations',
    'previous_bookings_not_canceled', 'required_car_parking_spaces',
    'total_of_special_requests', 'revenue_potential', 'booking_month',
    # Engineered
    'has_children', 'is_weekend_heavy', 'repeat_cancel_risk', 'is_repeated_guest',
    # Categorical → encoded below
    'hotel', 'meal', 'market_segment', 'distribution_channel',
    'reserved_room_type', 'deposit_type', 'customer_type',
]

TARGET = 'is_canceled'

df_ml = df[FEATURES + [TARGET]].copy()

# ─── Label-encode categoricals ──────────────────────────────────────────
CAT_COLS = ['hotel', 'meal', 'market_segment', 'distribution_channel',
            'reserved_room_type', 'deposit_type', 'customer_type']

le_dict = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df_ml[col] = le.fit_transform(df_ml[col].astype(str).str.strip().str.title())
    le_dict[col] = le

# ─── Show class balance ──────────────────────────────────────────────────
vc = df_ml[TARGET].value_counts()
print("Class balance:")
print(f"  Not Canceled (0): {vc[0]:,} ({vc[0]/len(df_ml)*100:.1f}%)")
print(f"  Canceled     (1): {vc[1]:,} ({vc[1]/len(df_ml)*100:.1f}%)")
print(f"\nTotal ML-ready rows: {len(df_ml):,}")
df_ml.head(3)


## 5. Train / Test Split

In [ ]:
X = df_ml.drop(columns=[TARGET])
y = df_ml[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Train size : {len(X_train):,} rows")
print(f"Test size  : {len(X_test):,} rows")
print(f"Feature cnt: {X_train.shape[1]}")


## 6. Model 1 — Random Forest Classifier
The primary model. Robust to outliers, handles mixed feature types, and provides 
native feature importance rankings — ideal for hotel booking data.


In [ ]:
# ─── Train ───────────────────────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators   = 300,
    max_depth      = 18,
    min_samples_leaf = 4,
    class_weight   = 'balanced',
    random_state   = 42,
    n_jobs         = -1
)
rf.fit(X_train, y_train)

# ─── Evaluate ────────────────────────────────────────────────────────────
y_pred_rf    = rf.predict(X_test)
y_proba_rf   = rf.predict_proba(X_test)[:, 1]

acc_rf  = accuracy_score(y_test, y_pred_rf)
auc_rf  = roc_auc_score(y_test, y_proba_rf)

print("=" * 55)
print("  RANDOM FOREST — RESULTS")
print("=" * 55)
print(f"  Accuracy : {acc_rf*100:.2f}%")
print(f"  ROC-AUC  : {auc_rf:.4f}")
print()
print(classification_report(y_test, y_pred_rf,
      target_names=['Not Canceled', 'Canceled']))


In [ ]:
# ─── Feature Importance (RF) ─────────────────────────────────────────────
feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [PALETTE[0] if i < 5 else '#B0BEC5' for i in range(len(feat_imp[:15]))]
feat_imp[:15].plot(kind='barh', ax=ax, color=colors[::-1])
ax.invert_yaxis()
ax.set_title('Random Forest — Top 15 Feature Importances', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Importance Score')
for bar, val in zip(ax.patches, feat_imp[:15].values[::-1]):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8.5)
plt.tight_layout()
plt.savefig('rf_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: rf_feature_importance.png")


## 7. Model 2 — Logistic Regression
Interpretable baseline model. Coefficients reveal the linear relationship between 
each feature and cancellation probability. Requires feature scaling.


In [ ]:
# ─── Scale + Train ───────────────────────────────────────────────────────
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

lr = LogisticRegression(
    max_iter     = 1000,
    class_weight = 'balanced',
    random_state = 42,
    C            = 1.0
)
lr.fit(X_train_sc, y_train)

y_pred_lr  = lr.predict(X_test_sc)
y_proba_lr = lr.predict_proba(X_test_sc)[:, 1]

acc_lr = accuracy_score(y_test, y_pred_lr)
auc_lr = roc_auc_score(y_test, y_proba_lr)

print("=" * 55)
print("  LOGISTIC REGRESSION — RESULTS")
print("=" * 55)
print(f"  Accuracy : {acc_lr*100:.2f}%")
print(f"  ROC-AUC  : {auc_lr:.4f}")
print()
print(classification_report(y_test, y_pred_lr,
      target_names=['Not Canceled', 'Canceled']))


In [ ]:
# ─── Coefficients plot ───────────────────────────────────────────────────
coef_df = pd.DataFrame({
    'feature': X.columns,
    'coef'   : lr.coef_[0]
}).sort_values('coef', key=abs, ascending=False).head(15)

colors_lr = [PALETTE[1] if c > 0 else PALETTE[0] for c in coef_df['coef']]
fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(coef_df['feature'][::-1], coef_df['coef'][::-1], color=colors_lr[::-1])
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_title('Logistic Regression — Top 15 Coefficients', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Coefficient (scaled)')
plt.tight_layout()
plt.savefig('lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: lr_coefficients.png")


## 8. Model 3 — Decision Tree (Interpretable Rules)

In [ ]:
dt = DecisionTreeClassifier(
    max_depth    = 7,
    class_weight = 'balanced',
    random_state = 42
)
dt.fit(X_train, y_train)

y_pred_dt  = dt.predict(X_test)
y_proba_dt = dt.predict_proba(X_test)[:, 1]

acc_dt = accuracy_score(y_test, y_pred_dt)
auc_dt = roc_auc_score(y_test, y_proba_dt)

print("=" * 55)
print("  DECISION TREE — RESULTS")
print("=" * 55)
print(f"  Accuracy : {acc_dt*100:.2f}%")
print(f"  ROC-AUC  : {auc_dt:.4f}")
print()
print(classification_report(y_test, y_pred_dt,
      target_names=['Not Canceled', 'Canceled']))

# ─── Top decision rules (text) ────────────────────────────────────────────
print("\n── Top Decision Rules (depth ≤ 3) ──")
print(export_text(dt, feature_names=list(X.columns), max_depth=3))


## 9. Model 4 — XGBoost (Gradient Boosting)

In [ ]:
if XGBOOST_AVAILABLE:
    xgb = XGBClassifier(
        n_estimators    = 300,
        max_depth       = 6,
        learning_rate   = 0.05,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        scale_pos_weight= (y_train == 0).sum() / (y_train == 1).sum(),
        random_state    = 42,
        eval_metric     = 'logloss',
        verbosity       = 0
    )
    xgb.fit(X_train, y_train,
            eval_set=[(X_test, y_test)],
            verbose=False)

    y_pred_xgb  = xgb.predict(X_test)
    y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

    acc_xgb = accuracy_score(y_test, y_pred_xgb)
    auc_xgb = roc_auc_score(y_test, y_proba_xgb)

    print("=" * 55)
    print("  XGBOOST — RESULTS")
    print("=" * 55)
    print(f"  Accuracy : {acc_xgb*100:.2f}%")
    print(f"  ROC-AUC  : {auc_xgb:.4f}")
    print()
    print(classification_report(y_test, y_pred_xgb,
          target_names=['Not Canceled', 'Canceled']))
else:
    print("XGBoost not installed. Install with: pip install xgboost")
    acc_xgb, auc_xgb = None, None
    y_proba_xgb = None


## 10. Model Comparison — ROC Curves & Accuracy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: ROC curves ─────────────────────────────────────────────────────
ax = axes[0]
models_roc = [
    ('Random Forest',      y_proba_rf,  PALETTE[0]),
    ('Logistic Regression',y_proba_lr,  PALETTE[1]),
    ('Decision Tree',      y_proba_dt,  PALETTE[2]),
]
if XGBOOST_AVAILABLE and y_proba_xgb is not None:
    models_roc.append(('XGBoost', y_proba_xgb, PALETTE[3]))

for name, proba, color in models_roc:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_val = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, color=color, lw=2.5, label=f'{name} (AUC={auc_val:.3f})')

ax.plot([0,1],[0,1],'--', color='#9E9E9E', linewidth=1.2, label='Random Guess')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves — All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([0,1]); ax.set_ylim([0,1.02])

# ── Right: Accuracy & AUC bars ───────────────────────────────────────────
ax2 = axes[1]
model_names   = ['Random Forest', 'Logistic Reg.', 'Decision Tree']
accuracies    = [acc_rf*100, acc_lr*100, acc_dt*100]
auc_scores    = [auc_rf, auc_lr, auc_dt]
if XGBOOST_AVAILABLE and acc_xgb is not None:
    model_names.append('XGBoost')
    accuracies.append(acc_xgb*100)
    auc_scores.append(auc_xgb)

x = np.arange(len(model_names))
w = 0.35
bars1 = ax2.bar(x - w/2, accuracies, w, label='Accuracy %', color=PALETTE[0], alpha=0.9)
bars2 = ax2.bar(x + w/2, [a*100 for a in auc_scores], w, label='ROC-AUC ×100', color=PALETTE[1], alpha=0.9)
ax2.set_xticks(x); ax2.set_xticklabels(model_names, fontsize=10)
ax2.set_ylabel('Score'); ax2.set_ylim([60, 105])
ax2.set_title('Accuracy & AUC Comparison', fontsize=14, fontweight='bold')
ax2.legend()
for bar in bars1: ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                            f'{bar.get_height():.1f}', ha='center', fontsize=9, fontweight='bold')
for bar in bars2: ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                            f'{bar.get_height():.1f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: model_comparison.png")


## 11. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cms = [
    (y_pred_rf,  'Random Forest',       PALETTE[0]),
    (y_pred_lr,  'Logistic Regression', PALETTE[1]),
    (y_pred_dt,  'Decision Tree',       PALETTE[2]),
]
for ax, (pred, name, color) in zip(axes, cms):
    cm = confusion_matrix(y_test, pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                   display_labels=['Not Canceled', 'Canceled'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('True Label')

plt.suptitle('Confusion Matrices — Test Set', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrices.png")


## 12. 5-Fold Stratified Cross-Validation

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Running 5-fold CV (may take ~60 s for Random Forest) …\n")

cv_results = {}
for name, model, Xd in [
    ('Random Forest',       rf, X),
    ('Logistic Regression', Pipeline([('sc', StandardScaler()), ('lr', lr)]), X),
    ('Decision Tree',       dt, X),
]:
    scores = cross_val_score(model, Xd, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    cv_results[name] = scores
    print(f"  {name:25s} AUC: {scores.mean():.4f} ± {scores.std():.4f}")

# ─── Plot CV distributions ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
data   = list(cv_results.values())
labels = list(cv_results.keys())
bp = ax.boxplot(data, labels=labels, patch_artist=True, widths=0.4,
                medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.85)
ax.set_ylabel('ROC-AUC (5-fold CV)')
ax.set_title('Cross-Validation Stability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('cv_results.png', dpi=150, bbox_inches='tight')
plt.show()


## 13. Cancellation Probability Scoring
Export a scored dataset with each booking's predicted cancellation probability — ready to power the live dashboard.


In [ ]:
# ─── Score the ENTIRE cleaned dataset with Random Forest ─────────────────
X_all = df_ml.drop(columns=[TARGET])
proba_all = rf.predict_proba(X_all)[:, 1]

df_scored = df.loc[df_ml.index].copy()
df_scored['cancel_probability']    = proba_all.round(4)
df_scored['cancel_risk_label']     = pd.cut(
    proba_all,
    bins   = [0, 0.30, 0.60, 1.0],
    labels = ['Low Risk', 'Medium Risk', 'High Risk']
)
df_scored['predicted_canceled']    = (proba_all >= 0.50).astype(int)

# ─── Summary ─────────────────────────────────────────────────────────────
print("Cancellation Risk Distribution:")
print(df_scored['cancel_risk_label'].value_counts())
print(f"\nHigh-Risk bookings  : {(df_scored['cancel_risk_label']=='High Risk').sum():,}")
print(f"Medium-Risk bookings: {(df_scored['cancel_risk_label']=='Medium Risk').sum():,}")
print(f"Low-Risk bookings   : {(df_scored['cancel_risk_label']=='Low Risk').sum():,}")

# ─── Export CSV for dashboard ─────────────────────────────────────────────
export_cols = [
    'hotel','arrival_date_year','arrival_date_month',
    'market_segment','deposit_type','customer_type',
    'lead_time','adr','total_stay','total_guests',
    'is_canceled','cancel_probability','cancel_risk_label','predicted_canceled'
]
df_scored[export_cols].to_csv('hotel_ml_scored.csv', index=False)
print("\n✅ Scored dataset saved: hotel_ml_scored.csv")


## 14. Risk Distribution Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Risk label distribution ───────────────────────────────────────────────
ax = axes[0]
rc = df_scored['cancel_risk_label'].value_counts()
colors_risk = [PALETTE[2], PALETTE[3], PALETTE[1]]
ax.bar(rc.index, rc.values, color=colors_risk, edgecolor='white', linewidth=0.5)
ax.set_title('Bookings by Risk Category', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Bookings')
for bar in ax.patches:
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+200,
            f'{int(bar.get_height()):,}', ha='center', fontsize=10, fontweight='bold')

# ── Probability histogram by actual outcome ───────────────────────────────
ax2 = axes[1]
for label, color in [(0, PALETTE[0]), (1, PALETTE[1])]:
    subset = df_scored[df_scored['is_canceled'] == label]['cancel_probability']
    ax2.hist(subset, bins=40, alpha=0.65, color=color,
             label='Not Canceled' if label==0 else 'Canceled', density=True)
ax2.set_xlabel('Predicted Cancellation Probability')
ax2.set_ylabel('Density')
ax2.set_title('Probability Distribution by Actual Outcome', fontsize=13, fontweight='bold')
ax2.legend()
ax2.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold 0.5')

# ── Risk by market segment ────────────────────────────────────────────────
ax3 = axes[2]
seg_risk = df_scored.groupby('market_segment')['cancel_probability'].mean().sort_values(ascending=True)
seg_risk.plot(kind='barh', ax=ax3, color=PALETTE[0])
ax3.set_title('Mean Cancel Probability
by Market Segment', fontsize=13, fontweight='bold')
ax3.set_xlabel('Mean Cancellation Probability')
for bar, val in zip(ax3.patches, seg_risk.values):
    ax3.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
             f'{val:.2f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('risk_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: risk_distributions.png")


## 15. Final Model Summary & Results Table

In [ ]:
results = {
    'Model'   : ['Random Forest', 'Logistic Regression', 'Decision Tree'],
    'Accuracy': [f'{acc_rf*100:.2f}%', f'{acc_lr*100:.2f}%', f'{acc_dt*100:.2f}%'],
    'ROC-AUC' : [f'{auc_rf:.4f}',      f'{auc_lr:.4f}',      f'{auc_dt:.4f}'],
    'Notes'   : [
        '🥇 Best overall — use for production scoring',
        'Interpretable baseline — coefficients tell the story',
        'Human-readable rules — good for presentations'
    ]
}
if XGBOOST_AVAILABLE and acc_xgb is not None:
    results['Model'].append('XGBoost')
    results['Accuracy'].append(f'{acc_xgb*100:.2f}%')
    results['ROC-AUC'].append(f'{auc_xgb:.4f}')
    results['Notes'].append('Gradient boosting — high performance')

summary_df = pd.DataFrame(results)
print("=" * 75)
print("  FINAL MODEL COMPARISON")
print("=" * 75)
print(summary_df.to_string(index=False))
print()
print("📦 Output Files:")
print("  hotel_ml_scored.csv        ← scored dataset for dashboard")
print("  rf_feature_importance.png  ← feature importance chart")
print("  model_comparison.png       ← ROC + accuracy bar chart")
print("  confusion_matrices.png     ← confusion matrix grid")
print("  cv_results.png             ← cross-validation boxplot")
print("  risk_distributions.png     ← risk category visualizations")


---
## ✅ Notebook Complete

| File | Use |
|------|-----|
| `hotel_ml_scored.csv` | Upload to your dashboard site |
| `rf_feature_importance.png` | Add to the ML section of the site |
| `model_comparison.png` | Add to the presentation slide |
| `confusion_matrices.png` | Add to the EDA report |

**Next Steps:**
1. Run this notebook with the real `hotel_booking.csv`
2. Upload `hotel_ml_scored.csv` to the site's data folder
3. The upgraded site HTML below reads `cancel_probability` from the CSV to power the live ML predictor widget
